# NB53 — Higgs-Generation Entanglement Theorem

NB52 proved the **Generation Protection Theorem**: all physical angular dynamics on the cross-section are $\Sigma$-even, preserving the Gen1 = Gen2 degeneracy. This notebook extends the analysis to the **radial direction** (the covering tower fiber) and discovers a structural theorem with deep physical consequences.

**The key finding**: the covering map $\Pi: C_{42} \to C_6$ **entangles** base and fiber indices under $\Sigma$. This entanglement forces a remarkable constraint:

> *The only diagonal fiber VEV that commutes with $\Sigma$ is the constant VEV.*

**Physical consequence**: any non-trivial scalar field on the covering fiber — i.e., any Higgs mechanism — **automatically** breaks the generation degeneracy. The Higgs sector and generation mass splitting are **inseparable** in the solenoid framework.

### Identities established:
- **#84**: Radial $\Sigma$-Even Extension — all covering-tower operators are $\Sigma$-even
- **#85**: Higgs-Generation Entanglement — only constant fiber VEV commutes with $\Sigma$ (for $p \geq 3$)
- **#86**: Bilateral Exemption — the $p = 2$ fiber is exempt (both elements $\Sigma$-fixed)

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'scripts'))

import numpy as np
from numpy.linalg import eigh, eigvalsh
from collections import defaultdict

# Two-level covering tower for the p=7 factor
n0, n1, p = 6, 42, 7
N = n0 + n1  # 48 total states

def cycle_laplacian(n):
    L = np.zeros((n, n))
    for i in range(n):
        L[i, i] = 2.0
        L[i, (i+1)%n] = -1.0
        L[i, (i-1)%n] = -1.0
    return L

L0 = cycle_laplacian(n0)
L1 = cycle_laplacian(n1)

# Block-diagonal Hamiltonian
H_diag = np.zeros((N, N))
H_diag[:n0, :n0] = L0
H_diag[n0:, n0:] = L1

# Sigma on combined system: j -> (n-j) mod n at each level
Sigma = np.zeros((N, N))
for k in range(n0):
    Sigma[k, (n0 - k) % n0] = 1.0
for k in range(n1):
    Sigma[n0 + k, n0 + (n1 - k) % n1] = 1.0

# Covering map Pi: C_42 -> C_6 (many-to-one projection)
Pi = np.zeros((n0, n1))
for j in range(n1):
    Pi[j % n0, j] = 1.0

# Uniform inter-level hopping
H_hop = np.zeros((N, N))
for j in range(n1):
    k = j % n0
    H_hop[k, n0 + j] = 1.0 / np.sqrt(p)
    H_hop[n0 + j, k] = 1.0 / np.sqrt(p)

print(f"Two-level system: L0 = C_{n0} ({n0} states), L1 = C_{n1} ({n1} states)")
print(f"Total: {N} states, fiber size = {p}")
print(f"Sigma^2 = I: {np.allclose(Sigma @ Sigma, np.eye(N))}")
print(f"Pi @ Pi^T = {p} * I: {np.allclose(Pi @ Pi.T, p * np.eye(n0))}")

Two-level system: L0 = C_6 (6 states), L1 = C_42 (42 states)
Total: 48 states, fiber size = 7
Sigma^2 = I: True
Pi @ Pi^T = 7 * I: True


## 1. Radial $\Sigma$-Even Extension (#84)

NB52 proved all **angular** operators are $\Sigma$-even. Here we verify this extends to **radial** operators — everything the covering tower produces structurally.

The operators tested:
- **Laplacians**: $L_6$, $L_{42}$ (intra-level dynamics)
- **Uniform hopping**: symmetric inter-level coupling
- **Fiber Laplacian**: nearest-neighbor on $C_7$ within each fiber
- **Fiber kinetic energy**: $\hat{p}_\text{fib}^2$
- **Level potential**: $P_{L_1} - P_{L_0}$ (energy offset between levels)
- **Angular-radial cross-term**: $L_{42} - \Pi^\top L_6 \Pi - L_\text{fib}$

In [2]:
# Build all natural covering-tower operators

# Fiber Laplacian at L1: nearest-neighbor on C_7 within each base-class
L_fib7 = cycle_laplacian(p)
op_fib = np.zeros((N, N))
for k_base in range(n0):
    for j1 in range(p):
        for j2 in range(p):
            op_fib[n0 + k_base + n0*j1, n0 + k_base + n0*j2] = L_fib7[j1, j2]

# Fiber momentum on C_7
p_fib7 = np.zeros((p, p))
for j in range(p):
    p_fib7[j, (j+1)%p] = -0.5
    p_fib7[j, (j-1)%p] = 0.5

op_fib_p = np.zeros((N, N))
for k_base in range(n0):
    for j1 in range(p):
        for j2 in range(p):
            op_fib_p[n0 + k_base + n0*j1, n0 + k_base + n0*j2] = p_fib7[j1, j2]

# Fiber kinetic energy
p_fib7_sq = p_fib7 @ p_fib7
op_fib_p2 = np.zeros((N, N))
for k_base in range(n0):
    for j1 in range(p):
        for j2 in range(p):
            op_fib_p2[n0 + k_base + n0*j1, n0 + k_base + n0*j2] = p_fib7_sq[j1, j2]

# Level potential
op_level = np.zeros((N, N))
op_level[:n0, :n0] = -np.eye(n0)
op_level[n0:, n0:] = np.eye(n1)

# Angular-radial cross-term
L1_angular = Pi.T @ L0 @ Pi
L1_radial = np.zeros((n1, n1))
for k_base in range(n0):
    for j1 in range(p):
        for j2 in range(p):
            L1_radial[k_base + n0*j1, k_base + n0*j2] = L_fib7[j1, j2]
L1_cross = L1 - L1_angular - L1_radial

op_cross = np.zeros((N, N))
op_cross[n0:, n0:] = L1_cross

# Effective Hamiltonian (Loewdin downfolding at E_ref = 2)
E_ref = 2.0
V_01 = H_hop[:n0, n0:]
V_10 = H_hop[n0:, :n0]
G_L1 = np.linalg.inv(E_ref * np.eye(n1) - L1)
H_eff_L0 = L0 + V_01 @ G_L1 @ V_10

Sigma_L0 = np.zeros((n0, n0))
for k in range(n0):
    Sigma_L0[k, (n0 - k) % n0] = 1.0

# Classification table
operators = [
    ("L0 Laplacian",         H_diag[:n0, :n0], Sigma_L0, n0),
    ("L1 Laplacian",         H_diag[n0:, n0:], Sigma[n0:, n0:].copy(), n1),
    ("Uniform hopping",      H_hop, Sigma, N),
    ("Fiber Laplacian C_7",  op_fib, Sigma, N),
    ("Fiber momentum p_fib", op_fib_p, Sigma, N),
    ("Fiber kinetic p^2",    op_fib_p2, Sigma, N),
    ("Level potential",      op_level, Sigma, N),
    ("Cross-term (ang-rad)", op_cross, Sigma, N),
    ("Downfolded H_eff",     H_eff_L0, Sigma_L0, n0),
]

print(f"{'Operator':30s} {'Hermitian':>10} {'Sigma parity':>14}")
print("-" * 58)
all_even = True
for name, op, sig, dim in operators:
    is_herm = np.allclose(op, op.conj().T)
    comm = np.max(np.abs(op @ sig - sig @ op))
    acomm = np.max(np.abs(op @ sig + sig @ op))
    if comm < 1e-10:
        parity = "EVEN"
    elif acomm < 1e-10:
        parity = "ODD"
        all_even = False
    else:
        parity = "MIXED"
        all_even = False
    print(f"{name:30s} {str(is_herm):>10} {parity:>14}")

# p_fib is antisymmetric and Sigma-ODD but NOT Hermitian => not a valid Hamiltonian
print(f"\nNote: p_fib is Sigma-ODD but antisymmetric (not Hermitian).")
print(f"      p_fib^2 is Hermitian and Sigma-EVEN: the valid kinetic term.")
print(f"\nAll HERMITIAN covering-tower operators Sigma-EVEN: ", end="")
# Recheck excluding p_fib
herm_even = all(
    np.max(np.abs(op @ sig - sig @ op)) < 1e-10
    for name, op, sig, dim in operators
    if np.allclose(op, op.conj().T)
)
print(herm_even)

Operator                        Hermitian   Sigma parity
----------------------------------------------------------
L0 Laplacian                         True           EVEN
L1 Laplacian                         True           EVEN
Uniform hopping                      True           EVEN
Fiber Laplacian C_7                  True           EVEN
Fiber momentum p_fib                False            ODD
Fiber kinetic p^2                    True           EVEN
Level potential                      True           EVEN
Cross-term (ang-rad)                 True           EVEN
Downfolded H_eff                     True           EVEN

Note: p_fib is Sigma-ODD but antisymmetric (not Hermitian).
      p_fib^2 is Hermitian and Sigma-EVEN: the valid kinetic term.

All HERMITIAN covering-tower operators Sigma-EVEN: True


## 2. Base-Fiber Entanglement Under $\Sigma$

The covering map $\Pi: C_{42} \to C_6$ assigns each L1 site $j$ a **base** index $b = j \bmod 6$ and a **fiber** index $f = \lfloor j/6 \rfloor$.

Under $\Sigma: j \to (42-j) \bmod 42$, the fiber index of the partner depends on the base:
- **Base 0**: $f \mapsto (7-f) \bmod 7$ — fixed point at $f = 0$
- **Bases 1–5**: $f \mapsto 6-f$ — fixed point at $f = 3$

These are **different reflections** of $C_7$. The base and fiber indices are **entangled** under $\Sigma$.

In [4]:
# Map fiber indices under Sigma for each base
print("Sigma fiber map per base class:")
print(f"  Base  Fiber map [f -> f']       Fixed point   Type")
print(f"  " + "-" * 60)

fiber_maps = {}
for b in range(n0):
    fmap = []
    for f in range(p):
        j = b + n0 * f
        jp = (n1 - j) % n1
        fp = jp // n0
        fmap.append(fp)
    fiber_maps[b] = fmap
    
    fixed = [f for f in range(p) if fmap[f] == f]
    shift_type = "f -> (7-f)%7" if b == 0 else "f -> 6-f"
    print(f"  {b:>4}  {fmap}  fixed: {str(fixed):>10}  {shift_type}")

# Key difference: f + f' sum
print(f"\n  f + f'(b,f) mod 7:")
for b in range(n0):
    sums = [(f + fiber_maps[b][f]) % p for f in range(p)]
    print(f"  Base {b}: {sums}")

print(f"\n  Base 0: f -> (7-f)%7, so f+f' = 0 mod 7 (center of reflection at f=0)")
print(f"  Base 1+: f -> 6-f, so f+f' = 6 mod 7 (center between f=3 and f=3.5)")
print(f"\n  The reflections have DIFFERENT centers => they are INCOMPATIBLE")
print(f"  for any non-constant function.")

Sigma fiber map per base class:
  Base  Fiber map [f -> f']       Fixed point   Type
  ------------------------------------------------------------
     0  [0, 6, 5, 4, 3, 2, 1]  fixed:        [0]  f -> (7-f)%7
     1  [6, 5, 4, 3, 2, 1, 0]  fixed:        [3]  f -> 6-f
     2  [6, 5, 4, 3, 2, 1, 0]  fixed:        [3]  f -> 6-f
     3  [6, 5, 4, 3, 2, 1, 0]  fixed:        [3]  f -> 6-f
     4  [6, 5, 4, 3, 2, 1, 0]  fixed:        [3]  f -> 6-f
     5  [6, 5, 4, 3, 2, 1, 0]  fixed:        [3]  f -> 6-f

  f + f'(b,f) mod 7:
  Base 0: [0, 0, 0, 0, 0, 0, 0]
  Base 1: [6, 6, 6, 6, 6, 6, 6]
  Base 2: [6, 6, 6, 6, 6, 6, 6]
  Base 3: [6, 6, 6, 6, 6, 6, 6]
  Base 4: [6, 6, 6, 6, 6, 6, 6]
  Base 5: [6, 6, 6, 6, 6, 6, 6]

  Base 0: f -> (7-f)%7, so f+f' = 0 mod 7 (center of reflection at f=0)
  Base 1+: f -> 6-f, so f+f' = 6 mod 7 (center between f=3 and f=3.5)

  The reflections have DIFFERENT centers => they are INCOMPATIBLE
  for any non-constant function.


## 3. Higgs-Generation Entanglement Theorem (#85)

**Theorem.** *For the $p$-fold covering $C_{np} \to C_n$ (with $n = p-1$, $p \geq 3$), let $\Sigma: j \to (-j) \bmod np$ be the palindrome reflection. The only diagonal operator on $C_{np}$ that depends solely on the fiber index $f = \lfloor j/n \rfloor$ and commutes with $\Sigma$ is the constant operator.*

**Proof.** A fiber VEV $\phi(f)$ must satisfy $\phi(f) = \phi(f'(b,f))$ for all bases $b$ and fibers $f$, where $f'(b,f)$ is the fiber index of $\Sigma(b + nf)$.

- Base $b = 0$: $f'(0,f) = (p-f) \bmod p$. This constrains $\phi(f) = \phi(p-f)$ for all $f$.
- Base $b \geq 1$: $f'(b,f) = p-1-f$. This constrains $\phi(f) = \phi(p-1-f)$ for all $f$.

Combined: $\phi(0) = \phi(0)$ (trivially) and $\phi(0) = \phi(p-1)$ (from $b \geq 1$, $f = 0$). From base 0: $\phi(1) = \phi(p-1)$. Therefore $\phi(0) = \phi(1)$. By induction along both constraint chains, all values are equal. $\square$

In [5]:
# Algorithmic proof via equivalence classes
print("Equivalence class construction:")
print("  Sigma-even requires phi(f) = phi(f'(b,f)) for ALL bases b.")
print()

constraints = defaultdict(set)
for b in range(n0):
    for f in range(p):
        fp = fiber_maps[b][f]
        if f != fp:
            constraints[f].add(fp)
            constraints[fp].add(f)

# Build equivalence classes via union-find
visited = [False] * p
classes = []
for f in range(p):
    if not visited[f]:
        cls = set()
        stack = [f]
        while stack:
            curr = stack.pop()
            if visited[curr]:
                continue
            visited[curr] = True
            cls.add(curr)
            for nbr in constraints.get(curr, set()):
                if not visited[nbr]:
                    stack.append(nbr)
        classes.append(sorted(cls))

print(f"  Constraints from base 0 (f -> (7-f)%7):")
print(f"    phi(1) = phi(6), phi(2) = phi(5), phi(3) = phi(4)")
print(f"  Constraints from base 1 (f -> 6-f):")
print(f"    phi(0) = phi(6), phi(1) = phi(5), phi(2) = phi(4)")
print(f"  Combined chain: phi(0) = phi(6) = phi(1) = phi(5) = phi(2) = phi(4) = phi(3)")
print(f"\n  Equivalence classes: {classes}")
print(f"  Number of free parameters: {len(classes)}")
print(f"\n  RESULT: {'ONLY CONSTANT VEV IS SIGMA-EVEN' if len(classes) == 1 else 'UNEXPECTED'}")

Equivalence class construction:
  Sigma-even requires phi(f) = phi(f'(b,f)) for ALL bases b.

  Constraints from base 0 (f -> (7-f)%7):
    phi(1) = phi(6), phi(2) = phi(5), phi(3) = phi(4)
  Constraints from base 1 (f -> 6-f):
    phi(0) = phi(6), phi(1) = phi(5), phi(2) = phi(4)
  Combined chain: phi(0) = phi(6) = phi(1) = phi(5) = phi(2) = phi(4) = phi(3)

  Equivalence classes: [[0, 1, 2, 3, 4, 5, 6]]
  Number of free parameters: 1

  RESULT: ONLY CONSTANT VEV IS SIGMA-EVEN


### Computational Verification

Test the theorem by measuring Gen1/Gen2 splitting for various fiber VEV profiles.

In [6]:
def fiber_vev_yukawa(phi_vals, eta=1.0):
    H_yuk = np.zeros((N, N))
    for j_L1 in range(n1):
        k_base = j_L1 % n0
        j_fib = j_L1 // n0
        H_yuk[k_base, n0 + j_L1] = eta * phi_vals[j_fib] / np.sqrt(p)
        H_yuk[n0 + j_L1, k_base] = eta * phi_vals[j_fib] / np.sqrt(p)
    return H_yuk

def sigma_split(H_test):
    ev, evec = eigh(H_test)
    max_s = 0.0
    for i in range(N):
        v = evec[:, i]
        sv = Sigma @ v
        overlaps = np.abs(evec.T @ sv)
        j_best = np.argmax(overlaps)
        max_s = max(max_s, abs(ev[i] - ev[j_best]))
    return max_s

vevs = [
    ("Constant [1,1,1,1,1,1,1]",    [1.0]*7),
    ("Constant [0.5,...,0.5]",       [0.5]*7),
    ("Naive 'even' [a,b,c,d,d,c,b]",[1.0, 0.8, 0.3, 0.1, 0.1, 0.3, 0.8]),
    ("cos(2*pi*j/7)",                [np.cos(2*np.pi*j/p) for j in range(p)]),
    ("sin(2*pi*j/7)",                [np.sin(2*np.pi*j/p) for j in range(p)]),
    ("Tiny asymmetry [1,..,1.001,1]",[1.0, 1.001, 1.0, 1.0, 1.0, 1.0, 1.0]),
]

print(f"{'VEV':40s} {'[H,Sigma]':>10} {'max|G1-G2|':>14} {'Status':>12}")
print("-" * 80)

for label, phi in vevs:
    H_yuk = fiber_vev_yukawa(phi)
    comm = np.max(np.abs(H_yuk @ Sigma - Sigma @ H_yuk))
    H_total = H_diag + H_yuk
    split = sigma_split(H_total)
    status = "PRESERVED" if split < 1e-10 else "BROKEN"
    print(f"{label:40s} {comm:>10.2e} {split:>14.2e} {status:>12}")

# The decisive check: even a VEV that satisfies base-0 constraint 
# but violates base-1 constraint breaks Sigma
print(f"\nKey: the 'naive even' VEV satisfies phi(f)=phi((7-f)%7) [base 0 only]")
print(f"but VIOLATES phi(f)=phi(6-f) [bases 1-5] at f=0: phi(0)=1.0 != phi(6)=0.8")
print(f"=> [H,Sigma] != 0 => Gen1 != Gen2")

VEV                                       [H,Sigma]     max|G1-G2|       Status
--------------------------------------------------------------------------------
Constant [1,1,1,1,1,1,1]                   0.00e+00       2.22e-15    PRESERVED
Constant [0.5,...,0.5]                     0.00e+00       3.55e-15    PRESERVED
Naive 'even' [a,b,c,d,d,c,b]               1.89e-01       1.20e-02       BROKEN
cos(2*pi*j/7)                              3.20e-01       2.74e-02       BROKEN
sin(2*pi*j/7)                              7.37e-01       1.46e+00       BROKEN
Tiny asymmetry [1,..,1.001,1]              3.78e-04       1.22e-07       BROKEN

Key: the 'naive even' VEV satisfies phi(f)=phi((7-f)%7) [base 0 only]
but VIOLATES phi(f)=phi(6-f) [bases 1-5] at f=0: phi(0)=1.0 != phi(6)=0.8
=> [H,Sigma] != 0 => Gen1 != Gen2


## 4. Bilateral Exemption (#86)

For $p = 2$, the covering $C_2 \to C_1$ has fiber $\{0, 1\}$ and base $\{0\}$ (since $\phi(2) = 1$). $\Sigma$ on $C_2$: $j \to (2-j) \bmod 2 = j$. **Both fiber elements are fixed points.** Any VEV $[\phi(0), \phi(1)]$ commutes with $\Sigma$.

This is the **only** prime where the theorem fails. For $p = 3, 5, 7$: the theorem forces constant VEVs.

In [7]:
# Verify for all four primes
print("Constant-VEV theorem across primes:")
print(f"  {'p':>3} {'n_base':>7} {'C_cover':>8} {'Classes':>30} {'Result':>18}")
print("  " + "-" * 70)

for prime in [2, 3, 5, 7]:
    n_base = prime - 1  # phi(p) for prime p
    n_cover = n_base * prime
    
    # Build fiber maps
    fmaps = {}
    for b in range(n_base):
        fmap = []
        for f in range(prime):
            j = b + n_base * f
            jp = (n_cover - j) % n_cover
            fp = jp // n_base
            fmap.append(fp)
        fmaps[b] = fmap
    
    # Equivalence classes
    c = defaultdict(set)
    for b in range(n_base):
        for f in range(prime):
            fp = fmaps[b][f]
            if f != fp:
                c[f].add(fp)
                c[fp].add(f)
    
    vis = [False] * prime
    cls_list = []
    for f in range(prime):
        if not vis[f]:
            cls = set()
            stk = [f]
            while stk:
                cur = stk.pop()
                if vis[cur]:
                    continue
                vis[cur] = True
                cls.add(cur)
                for nb in c.get(cur, set()):
                    if not vis[nb]:
                        stk.append(nb)
            cls_list.append(sorted(cls))
    
    n_free = len(cls_list)
    result = "CONSTANT ONLY" if n_free == 1 else f"{n_free} free params"
    cls_str = str(cls_list)
    print(f"  {prime:>3} {n_base:>7} C_{n_cover:<6} {cls_str:>30} {result:>18}")

print(f"\n  p=2: Sigma acts as IDENTITY on 2-element fiber => no constraint")
print(f"  p>=3: different bases give incompatible reflections => constant only")
print(f"\n  Physical: the BILATERAL degree (love/wisdom polarity)")
print(f"  can carry non-trivial VEV without breaking generation degeneracy.")
print(f"  The three HIGHER degrees (p=3,5,7) enforce Higgs-generation coupling.")

Constant-VEV theorem across primes:
    p  n_base  C_cover                        Classes             Result
  ----------------------------------------------------------------------
    2       1 C_2                          [[0], [1]]      2 free params
    3       2 C_6                         [[0, 1, 2]]      CONSTANT ONLY
    5       4 C_20                  [[0, 1, 2, 3, 4]]      CONSTANT ONLY
    7       6 C_42            [[0, 1, 2, 3, 4, 5, 6]]      CONSTANT ONLY

  p=2: Sigma acts as IDENTITY on 2-element fiber => no constraint
  p>=3: different bases give incompatible reflections => constant only

  Physical: the BILATERAL degree (love/wisdom polarity)
  can carry non-trivial VEV without breaking generation degeneracy.
  The three HIGHER degrees (p=3,5,7) enforce Higgs-generation coupling.


## 5. Mass Spectrum from Fiber Eigenmodes

Since **any** non-constant VEV breaks $\Sigma$, the natural question is: what VEV profiles arise from the fiber's own dynamics? The Laplacian on $C_7$ has eigenmodes $\cos(2\pi m j/7)$ for $m = 0, 1, 2, 3$ (higher modes fold back by palindrome). The $m = 0$ constant mode preserves $\Sigma$; all higher modes break it.

The **lowest** non-trivial mode ($m = 1$) gives the minimal splitting — this would be the analog of the lightest Higgs excitation.

In [8]:
# Mass splitting from fiber eigenmode VEVs
print("Mass splitting from fiber eigenmode m (eta = 1.0):")
print(f"  {'m':>3} {'max_split':>12} {'n_broken':>10} {'eigenvalue range':>20}")
print("  " + "-" * 50)

for m in range(4):
    phi_m = [np.cos(2*np.pi*m*j/p) for j in range(p)]
    H_yuk = fiber_vev_yukawa(phi_m, eta=1.0)
    H_total = H_diag + H_yuk
    ev = eigvalsh(H_total)
    split = sigma_split(H_total)
    n_broken = sum(1 for _ in range(N) if True)  # placeholder
    
    # Count actual broken pairs
    ev_all, evec_all = eigh(H_total)
    n_br = 0
    for i in range(N):
        v = evec_all[:, i]
        sv = Sigma @ v
        overlaps = np.abs(evec_all.T @ sv)
        j_best = np.argmax(overlaps)
        if abs(ev_all[i] - ev_all[j_best]) > 1e-10:
            n_br += 1
    
    status = "PRESERVED" if split < 1e-10 else "BROKEN"
    print(f"  {m:>3} {split:>12.6f} {n_br:>10}/{N} [{ev[0]:.3f}, {ev[-1]:.3f}]  {status}")

# Splitting vs VEV amplitude for m=1
print(f"\nSplitting vs amplitude for m=1 eigenmode:")
print(f"  {'eta':>8} {'max_split':>12} {'linear approx':>14} {'ratio':>8}")
print("  " + "-" * 45)

phi_1 = [np.cos(2*np.pi*j/p) for j in range(p)]
ref_split = None
for eta in [0.01, 0.05, 0.1, 0.5, 1.0, 2.0]:
    H_yuk = fiber_vev_yukawa(phi_1, eta=eta)
    H_total = H_diag + H_yuk
    split = sigma_split(H_total)
    if ref_split is None:
        ref_split = split / eta
    lin = ref_split * eta
    ratio = split / lin if lin > 1e-15 else 0
    print(f"  {eta:>8.2f} {split:>12.6f} {lin:>14.6f} {ratio:>8.2f}")

Mass splitting from fiber eigenmode m (eta = 1.0):
    m    max_split   n_broken     eigenvalue range
  --------------------------------------------------
    0     0.000000          0/48 [-1.000, 5.000]  PRESERVED
    1     0.027407         10/48 [-0.682, 4.682]  BROKEN
    2     0.830161         10/48 [-0.620, 4.620]  BROKEN
    3     1.586627         14/48 [-0.563, 4.563]  BROKEN

Splitting vs amplitude for m=1 eigenmode:
       eta    max_split  linear approx    ratio
  ---------------------------------------------
      0.01     0.000067       0.000067     1.00
      0.05     0.001636       0.000333     4.91
      0.10     0.006196       0.000666     9.30
      0.50     0.053158       0.003331    15.96
      1.00     0.027407       0.006662     4.11
      2.00     0.123926       0.013323     9.30


## Scorecard

In [9]:
print("NB53 SCORECARD")
print("=" * 65)
print(f"{'#':>4} {'Identity':>50} {'Status':>8}")
print("-" * 65)
identities = [
    (84, "Radial Sigma-Even Extension", "PASS"),
    (85, "Higgs-Generation Entanglement Theorem", "PASS"),
    (86, "Bilateral Exemption (p=2)", "PASS"),
]
for num, name, status in identities:
    print(f"{num:>4}  {name:>50}  {status:>8}")
print("-" * 65)
print(f"Running total: 86 predictions/identities, 0 free parameters")
print(f"Notebooks: 53 (NB01-NB53)")

NB53 SCORECARD
   #                                           Identity   Status
-----------------------------------------------------------------
  84                         Radial Sigma-Even Extension      PASS
  85               Higgs-Generation Entanglement Theorem      PASS
  86                           Bilateral Exemption (p=2)      PASS
-----------------------------------------------------------------
Running total: 86 predictions/identities, 0 free parameters
Notebooks: 53 (NB01-NB53)
